# POT-PTQ on a real ~1B-parameter LLM: reproduction + FP16 baseline

Reproduction of **"POT-PTQ: A Two-step Power-of-Two Post-training for LLMs"**
(Wang et al., arXiv:2507.11959) applied to a real ~1-billion-parameter
Hugging Face causal LM (`EleutherAI/pythia-1b` by default) instead of a toy
model, evaluated on the actual **WikiText-2** benchmark the paper uses
(Sec. 5.1–5.2, Table 1).

This notebook:
1. Loads a real ~1B model in FP16 and reports its **FP16 baseline
   perplexity on WikiText-2** (Table 1's `FP16`/uncompressed column).
2. Implements Algorithm 1 (data-agnostic scale init, Eq. 6–10), the
   PoT weight representation (Eq. 1–5), Algorithm 2 (data-dependent
   fine-tuning, Eq. 11–18, STE), and applies POT-PTQ to the model at
   **3-bit and 2-bit** precision, generalized to work with *any*
   Hugging Face decoder-only model (not just LLaMA), using a
   GPTQ-style layer-sequential calibration driver (forward-hook based,
   so it doesn't hard-code layer/module names).
3. Re-evaluates WikiText-2 perplexity for each quantized model and
   prints a **Table-1-style summary** (FP16 vs. 3-bit vs. 2-bit).
4. Runs the Step-1-only / Step-2-only / both ablation (Table 4 style)
   and the Section 4 bitwise-dequantization microbenchmark (Fig. 4),
   unchanged from the algorithmic core since those are architecture-
   independent.

**Requirements:** a CUDA GPU with ≥16GB memory is strongly recommended
(T4/A10G/A100 on Colab all work) and internet access to download the
model + WikiText-2 from the Hugging Face Hub. It will run on CPU but
will be very slow for the full-size evaluation — reduce `EVAL_SEQLEN`/
`EVAL_MAX_CHUNKS`/`CALIB_SEQS` in the config cell if so.

**Just run every cell in order** (`Runtime -> Run all` in Colab).


## 0. Setup

In [ ]:
# Installs the packages needed to load a real HF model + WikiText-2.
import importlib.util, subprocess, sys

def _ensure(pkg, pip_name=None):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg], check=True)

_ensure("torch")
_ensure("transformers")
_ensure("datasets")
_ensure("accelerate")

import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


## 1. Config

`MODEL_NAME` is any Hugging Face causal-LM checkpoint. The default,
`EleutherAI/pythia-1b`, is ~1.01B parameters, ungated, and downloads
without a HF token. Swap in another ~1B model (e.g.
`facebook/opt-1.3b`, `TinyLlama/TinyLlama_v1.1`) if you prefer — the
quantization driver below discovers the transformer block list and
`nn.Linear` layers generically, it does not hard-code LLaMA/GPT-NeoX
attribute names.


In [ ]:
# ---- model ----
MODEL_NAME = "EleutherAI/pythia-1b"     # ~1.01B params, ungated
MODEL_DTYPE = torch.float16

# ---- WikiText-2 evaluation (paper's benchmark, Sec. 5.2 / Table 1) ----
EVAL_SEQLEN     = 2048     # paper: 2048-token non-overlapping chunks
EVAL_MAX_CHUNKS = None     # GPU suggestion: None (use the whole test set).
                           # Set e.g. 40 to get a quick approximate number
                           # first on a slower GPU/CPU, then rerun with None.

# ---- POT-PTQ quantization (Sec. 5.1) ----
GROUP_SIZE     = 128      # paper uses 64 or 128
CALIB_SEQS     = 16       # paper: 128            GPU suggestion: 64-128
CALIB_SEQLEN   = 1024     # paper: 2048            GPU suggestion: 2048
EPOCHS_3BIT    = 3        # paper: 10               GPU suggestion: 10
EPOCHS_2BIT    = 5        # paper: 40               GPU suggestion: 20-40
QUANT_LR       = 1e-3     # matches paper Sec. 5.1
WEIGHT_DECAY   = 1e-1     # matches paper Sec. 5.1
CALIB_BATCH    = 1        # sequences per forward pass during calibration
                           # (raise if you have GPU memory to spare)

print("Config set. Model:", MODEL_NAME)


## 2. Load the model and tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=MODEL_DTYPE)
model.to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"{MODEL_NAME}: {n_params/1e9:.3f}B parameters, dtype={next(model.parameters()).dtype}")


## 3. Data: WikiText-2 (real benchmark, Sec. 5.1–5.2)

- **Eval set**: the WikiText-2 *test* split, concatenated and chunked
  into non-overlapping `EVAL_SEQLEN`-token windows — the standard
  protocol used by GPTQ/AWQ/OmniQuant and this paper's Table 1.
- **Calibration set**: `CALIB_SEQS` random contiguous windows of
  `CALIB_SEQLEN` tokens sampled from the *train* split, playing the
  role of the paper's "128 randomly sampled 2048-token sequences from
  WikiText-2" used for Step 2 (Sec. 5.1).


In [ ]:
from datasets import load_dataset

wikitext_test = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
wikitext_train = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

test_text = "\n\n".join(wikitext_test["text"])
train_text = "\n\n".join(wikitext_train["text"])

test_ids = tokenizer(test_text, return_tensors="pt").input_ids
train_ids = tokenizer(train_text, return_tensors="pt").input_ids
print(f"test tokens:  {test_ids.shape[1]:,}")
print(f"train tokens: {train_ids.shape[1]:,}")

def get_calibration_batches(train_ids, n_seqs, seq_len, device, seed=0):
    g = torch.Generator().manual_seed(seed)
    max_start = train_ids.shape[1] - seq_len - 1
    starts = torch.randint(0, max_start, (n_seqs,), generator=g)
    return [train_ids[:, s:s + seq_len].to(device) for s in starts]

calib_batches = get_calibration_batches(train_ids, CALIB_SEQS, CALIB_SEQLEN, DEVICE)
print(f"{len(calib_batches)} calibration sequences of length {CALIB_SEQLEN}")


## 4. WikiText-2 perplexity evaluator + **FP16 baseline**

Standard GPTQ-style evaluation: split the tokenized test set into
non-overlapping `seqlen`-token chunks, sum the per-chunk negative
log-likelihood (loss × seqlen), and exponentiate the token-weighted
average. This is the exact quantity Table 1 of the paper reports.


In [ ]:
@torch.no_grad()
def evaluate_wikitext2_ppl(model, test_ids, seqlen, device, max_chunks=None):
    model.eval()
    n_chunks = test_ids.shape[1] // seqlen
    if max_chunks is not None:
        n_chunks = min(n_chunks, max_chunks)
    nlls = []
    for i in range(n_chunks):
        batch = test_ids[:, i * seqlen:(i + 1) * seqlen].to(device)
        out = model(batch, labels=batch)
        nlls.append(out.loss.float() * seqlen)
        if device == "cuda":
            torch.cuda.empty_cache()
    ppl = torch.exp(torch.stack(nlls).sum() / (n_chunks * seqlen))
    return ppl.item(), n_chunks

fp16_ppl, n_chunks_used = evaluate_wikitext2_ppl(model, test_ids, EVAL_SEQLEN, DEVICE, EVAL_MAX_CHUNKS)
print(f"[FP16 baseline] WikiText-2 perplexity = {fp16_ppl:.3f}  ({n_chunks_used} chunks of {EVAL_SEQLEN} tokens)")


## 5. POT-PTQ core (Sec. 3, Eq. 1–18, Algorithm 1)

Unchanged math from the paper, made dtype-safe for FP16 models (the
grid search and STE rounding are computed internally in float32 for
numerical stability, then the reconstructed weight is cast back to the
model's working dtype for the forward pass).

Weight representation (Eq. 1 / Eq. 4): `W_tilde = S * P * 2^E`
- `S`: positive per-group scale (learnable residual in Step 2)
- `P`: sign, `P = sign(W)` (frozen after Step 1)
- `E`: exponent in `[0, qmax]`, `qmax = 2^(n-1) - 1` for `n`-bit PoT (Eq. 2)

Group-wise quantization (Eq. 5): each row of a linear layer's weight
matrix (`out_features x in_features`) is divided along the input
dimension into contiguous chunks of `group_size` ("each column of
`W^(l)` is divided into fixed-size groups (e.g., 64 or 128 rows)" —
transposed here since `nn.Linear.weight` is `(out, in)`).


In [ ]:
import torch.nn as nn
import torch.nn.functional as F


# ---------------- Straight-Through Estimator (Eq. 18) ----------------
def ste_round_clamp(x: torch.Tensor, qmax: int) -> torch.Tensor:
    '''round(clamp(x)) in forward, identity gradient in backward
    (Straight-Through Estimator, used for Eq. 16/18).'''
    rounded = torch.clamp(torch.round(x), 0, qmax)
    return x + (rounded - x).detach()


def qmax_for_bits(n_bits: int) -> int:
    '''qmax = 2^(n-1) - 1, Eq. (2).'''
    return 2 ** (n_bits - 1) - 1


# ---------------- group reshaping helpers (Eq. 5) ----------------
def to_groups(W: torch.Tensor, group_size: int):
    out_f, in_f = W.shape
    pad = (-in_f) % group_size
    if pad:
        W = F.pad(W, (0, pad))
    Wg = W.view(out_f, -1, group_size)
    return Wg, pad


def from_groups(Wg: torch.Tensor, pad: int, in_f: int):
    W = Wg.reshape(Wg.shape[0], -1)
    if pad:
        W = W[:, :in_f]
    return W


# ---------------- Algorithm 1: Parallel Data-Agnostic Scale Init ----------------
@torch.no_grad()
def data_agnostic_scale_init(W: torch.Tensor, n_bits: int, group_size: int = 128,
                              n_candidates: int = 200, device=None):
    '''Vectorized Algorithm 1: for every group, grid-search multipliers
    B = {0.01*i | i=1..n_candidates} of the base scale
    s0 = max|W_group| / (2^qmax - 1)                          (Eq. 10)
    to minimize the weight reconstruction MSE (Eq. 8/9), fully vectorized
    across all groups & candidates. Computed in float32 for numerical
    stability regardless of the model's working dtype.'''
    device = device or W.device
    qmax = qmax_for_bits(n_bits)
    Wf = W.to(device=device, dtype=torch.float32)
    Wg, pad = to_groups(Wf, group_size)          # (O, G, g)
    absW = Wg.abs()
    P = torch.sign(Wg)
    P[P == 0] = 1.0

    s0 = absW.amax(dim=-1, keepdim=True) / max(qmax, 1)    # (O, G, 1), Eq.10
    s0 = torch.clamp(s0, min=1e-12)

    B = torch.arange(1, n_candidates + 1, device=device, dtype=torch.float32) * 0.01
    sb = s0.unsqueeze(-1) * B.view(1, 1, 1, -1)             # (O, G, 1, C)
    sb = torch.clamp(sb, min=1e-12)

    absW_ = absW.unsqueeze(-1)                              # (O, G, g, 1)
    E = torch.clamp(torch.round(torch.log2(absW_ / sb)), 0, qmax)  # Eq.6
    Wg_hat = sb * P.unsqueeze(-1) * torch.pow(2.0, E)        # Eq.7

    Q1 = ((Wg.unsqueeze(-1) - Wg_hat) ** 2).sum(dim=2)       # Eq.8
    best_idx = torch.argmin(Q1, dim=-1)

    b_star = B[best_idx]
    S = (s0.squeeze(-1) * b_star).unsqueeze(-1)               # Eq.9

    return S, P, pad


# ---------------- PoTLinear: drop-in nn.Linear replacement ----------------
class PoTLinear(nn.Module):
    '''Quantized linear layer. Holds frozen sign P (Eq.1), Step-1 scale S
    (Eq.9), and learnable residual Gamma (Eq.15) that Step 2 fine-tunes.
    Dequantized weight W_tilde = Shat * P * 2^E(Shat) recomputed on the
    fly (Eq. 16-17). Internal math is float32; the returned weight is
    cast back to `compute_dtype` (the model's working dtype) for F.linear.'''

    def __init__(self, linear: nn.Linear, n_bits: int, group_size: int = 128,
                 compute_dtype: torch.dtype = torch.float16):
        super().__init__()
        self.in_features = linear.in_features
        self.out_features = linear.out_features
        self.n_bits = n_bits
        self.group_size = group_size
        self.qmax = qmax_for_bits(n_bits)
        self.compute_dtype = compute_dtype

        W = linear.weight.data.clone().float()
        S, P, pad = data_agnostic_scale_init(W, n_bits, group_size)
        self.pad = pad
        self.register_buffer("W_orig_grouped", to_groups(W, group_size)[0])
        self.register_buffer("P", P)
        self.register_buffer("S", S)                 # Step-1 scale, frozen
        self.Gamma = nn.Parameter(torch.zeros_like(S))  # Eq.15, Step-2 learnable

        self.bias = None
        if linear.bias is not None:
            self.bias = nn.Parameter(linear.bias.data.clone())

    def dequantized_weight(self):
        Shat = self.S * (1.0 + self.Gamma)                        # Eq.15
        Shat_c = torch.clamp(Shat.abs(), min=1e-12) * torch.sign(Shat + 1e-12)
        absW = self.W_orig_grouped.abs()
        E = ste_round_clamp(torch.log2(absW / Shat_c.abs()), self.qmax)   # Eq.16 (+STE)
        Wg_hat = Shat_c * self.P * torch.pow(2.0, E)               # Eq.17
        W_hat = from_groups(Wg_hat, self.pad, self.in_features)
        return W_hat.to(self.compute_dtype)

    def forward(self, x):
        W_hat = self.dequantized_weight()
        bias = self.bias.to(self.compute_dtype) if self.bias is not None else None
        return F.linear(x, W_hat, bias)

    def freeze(self):
        '''Bake Gamma into S, reset Gamma to 0 (before quantizing the next
        block using this block's *quantized* output as input).'''
        with torch.no_grad():
            self.S.mul_(1.0 + self.Gamma)
            self.Gamma.zero_()

    def weight_only_mse(self):
        with torch.no_grad():
            W_hat = self.dequantized_weight().float()
            Wg_hat, _ = to_groups(W_hat, self.group_size)
            return ((Wg_hat - self.W_orig_grouped) ** 2).mean().item()

print("PoT quantization core defined.")


## 6. Generic, architecture-agnostic Step-2 driver (Algorithm 2, Eq. 11–14)

The demo notebook this is based on hard-coded LLaMA-style attention/MLP
attribute names. A real ~1B model (GPT-NeoX, OPT, LLaMA, ...) doesn't
share those names or forward signatures (rotary embeddings, fused QKV,
attention masks, KV-cache args, ...), so this driver is fully generic:

1. **`get_transformer_blocks(model)`** locates the `ModuleList` of
   decoder blocks for common HF architectures (GPT-NeoX, LLaMA/Mistral/
   Qwen-style, GPT-2/OPT-style), by attribute path.
2. **Hook-based "catcher" capture**: to get block *i*'s exact input
   `(args, kwargs)` — whatever the block's own architecture requires
   (hidden_states, attention_mask, position embeddings, cache
   position, ...) — a `forward_pre_hook` on block *i* records them and
   raises an exception that aborts the forward pass immediately, so we
   never pay for computing blocks after *i*. This is the same "Catcher"
   trick used in the official GPTQ/AWQ implementations, and it makes
   the driver work for *any* HF decoder-only model without special-
   casing its forward signature.
3. For each block, in order: capture inputs -> compute `Horig` with a
   frozen deep-copy of the *original* block (Eq. 14's `F^(l)(W^(l),X)`)
   -> replace every `nn.Linear` in the block with `PoTLinear` (Step 1
   runs automatically in its constructor) -> fine-tune only the
   `Gamma` parameters of this block with Adam to minimize
   `‖Horig - Hquant‖² + (λ/2)‖Γ‖²` (Eq. 14, STE gradient Eq. 18) ->
   freeze `Gamma` into `S` -> move to the next block (whose captured
   input will now reflect this block's *quantized* output, exactly the
   paper's sequential per-block calibration).


In [ ]:
import copy


class StopForward(Exception):
    pass


def get_transformer_blocks(model):
    '''Locate the ModuleList of decoder blocks for common HF architectures.'''
    candidate_paths = [
        "gpt_neox.layers",       # Pythia / GPT-NeoX
        "model.layers",          # LLaMA / Mistral / Qwen / OPT (newer)
        "transformer.h",         # GPT-2 / GPT-J style
        "model.decoder.layers",  # OPT (older layout)
        "transformer.blocks",    # MPT style
    ]
    for path in candidate_paths:
        obj = model
        ok = True
        for part in path.split("."):
            if hasattr(obj, part):
                obj = getattr(obj, part)
            else:
                ok = False
                break
        if ok and isinstance(obj, nn.ModuleList):
            return obj
    raise ValueError(
        "Could not locate the transformer block list for this architecture. "
        "Add its attribute path to `candidate_paths` above."
    )


def get_linear_submodules(block):
    '''All nn.Linear layers inside a block, by dotted name relative to it.'''
    return {name: mod for name, mod in block.named_modules() if isinstance(mod, nn.Linear)}


def set_submodule(root, dotted_name, new_module):
    parts = dotted_name.split(".")
    obj = root
    for p in parts[:-1]:
        obj = getattr(obj, p)
    setattr(obj, parts[-1], new_module)


@torch.no_grad()
def _detach_clone(x):
    '''Detach + clone every tensor in a captured arg/kwarg so it owns its
    own storage and carries no autograd history, and is immune to being
    mutated in place by a later call (see note on KV-caches below).'''
    if torch.is_tensor(x):
        return x.detach().clone()
    return x


def capture_block_io(model, block, calib_batches):
    '''Run `model(batch)` for every calibration batch, aborting right
    before `block` executes, and record the exact (args, kwargs) it was
    about to be called with.

    IMPORTANT: most HF causal LMs default to `use_cache=True`, which
    passes each block a *mutable, shared* KV-cache object (e.g. a
    `DynamicCache`) that gets appended to in place on every call. Since
    Step 2 calls `block(*args, **kwargs)` many times (every epoch x every
    calibration batch) reusing the *same* captured kwargs, a shared cache
    object would get mutated across calls and its stale internal tensors
    (tied to an already-freed autograd graph from a previous `.backward()`)
    would trigger "Trying to backward through the graph a second time".
    We disable the cache when capturing (`use_cache=False`, with a
    fallback for architectures that don't accept the kwarg) and also
    detach+clone every captured tensor, so each replay is fully
    independent and safe to backward through repeatedly.'''
    captured = []

    def hook(module, args, kwargs):
        args = tuple(_detach_clone(a) for a in args)
        kwargs = {k: _detach_clone(v) for k, v in kwargs.items()}
        captured.append((args, kwargs))
        raise StopForward()

    handle = block.register_forward_pre_hook(hook, with_kwargs=True)
    try:
        for batch in calib_batches:
            try:
                model(batch, use_cache=False)
            except TypeError:
                # architecture doesn't accept use_cache as a top-level kwarg
                try:
                    model(batch)
                except StopForward:
                    pass
            except StopForward:
                pass
    finally:
        handle.remove()
    return captured


def quantize_model(model, ref_model, calib_batches, n_bits, group_size=128,
                    epochs=10, lr=1e-3, weight_decay=1e-1, device="cuda",
                    compute_dtype=torch.float16, verbose=True):
    '''Quantize `model` in-place with POT-PTQ at `n_bits` precision,
    using `ref_model` (a frozen deep-copy of the original FP model) as
    the Eq.14 reconstruction target.'''
    blocks = get_transformer_blocks(model)
    ref_blocks = get_transformer_blocks(ref_model)
    stats = []

    for i, (block, ref_block) in enumerate(zip(blocks, ref_blocks)):
        io = capture_block_io(model, block, calib_batches)

        linears = get_linear_submodules(block)
        pot_layers = {}
        for name, lin in linears.items():
            pot = PoTLinear(lin, n_bits=n_bits, group_size=group_size,
                             compute_dtype=compute_dtype).to(device)
            pot_layers[name] = pot
            set_submodule(block, name, pot)

        params = [pot.Gamma for pot in pot_layers.values()]
        for p in params:
            p.requires_grad_(True)
        opt = torch.optim.Adam(params, lr=lr)

        loss_val = None
        for epoch in range(epochs):
            for args, kwargs in io:
                with torch.no_grad():
                    Horig = ref_block(*args, **kwargs)
                    Horig = Horig[0] if isinstance(Horig, tuple) else Horig

                Hquant = block(*args, **kwargs)
                Hquant = Hquant[0] if isinstance(Hquant, tuple) else Hquant

                rec_loss = F.mse_loss(Horig.float(), Hquant.float())
                reg = sum((p.float() ** 2).sum() for p in params)
                loss = rec_loss + (weight_decay / 2.0) * reg
                opt.zero_grad()
                loss.backward()
                opt.step()
                loss_val = rec_loss.item()

        for pot in pot_layers.values():
            pot.freeze()
            for p in pot.parameters():
                p.requires_grad_(False)

        w_mse = sum(pot.weight_only_mse() for pot in pot_layers.values()) / len(pot_layers)
        stats.append({"block": i, "recon_loss": loss_val, "weight_mse": w_mse})
        if verbose:
            print(f"  [block {i:2d}/{len(blocks)-1}] output-recon MSE={loss_val:.6e}  avg weight MSE={w_mse:.6e}")

        if device == "cuda":
            torch.cuda.empty_cache()

    return model, stats

print("Generic Step-2 driver defined.")


## 7. Run POT-PTQ at 3-bit and 2-bit, evaluate WikiText-2 perplexity (Table 1 style)

Quantizes a fresh copy of the FP16 model at each precision, evaluates
WikiText-2 perplexity the same way as the FP16 baseline in Section 4,
and prints a Table-1-style summary next to it.

**Note on memory:** this keeps a frozen FP16 reference copy of the
whole model (`ref_model`) alongside the model being quantized, plus
each `PoTLinear` keeps its original weight group tensor for the MSE
diagnostic — so peak memory is roughly 2.5-3x the base model size.
For a 1B model (~2GB in FP16) this is still comfortably under 16GB,
but reduce `CALIB_SEQS`/`CALIB_SEQLEN`/`CALIB_BATCH` first if you hit
an OOM on a smaller GPU.


In [ ]:
ref_model = copy.deepcopy(model).to(DEVICE)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

results = {"fp16": fp16_ppl}
quantized_models = {}

for n_bits, epochs in [(3, EPOCHS_3BIT), (2, EPOCHS_2BIT)]:
    print(f"\n===== POT-PTQ {n_bits}-bit (Step1 + Step2, {epochs} epochs) =====")
    qmodel = copy.deepcopy(model).to(DEVICE)
    qmodel, stats = quantize_model(qmodel, ref_model, calib_batches, n_bits=n_bits,
                                    group_size=GROUP_SIZE, epochs=epochs, lr=QUANT_LR,
                                    weight_decay=WEIGHT_DECAY, device=DEVICE,
                                    compute_dtype=MODEL_DTYPE, verbose=True)
    ppl, _ = evaluate_wikitext2_ppl(qmodel, test_ids, EVAL_SEQLEN, DEVICE, EVAL_MAX_CHUNKS)
    print(f"[POT-PTQ {n_bits}-bit]  WikiText-2 perplexity = {ppl:.3f}")
    results[f"pot_{n_bits}bit"] = ppl
    quantized_models[n_bits] = qmodel

print("\n===== Summary (Table 1 style) =====")
for k, v in results.items():
    print(f"{k:12s}: perplexity = {v:.3f}")


## 8. Ablation: Step 1 only vs. Step 2 only vs. both (Table 4 style)

Confirms the paper's central claim: Step 1 (robust init) and Step 2
(output-aligned fine-tuning) are complementary — each alone
underperforms using both together. Run at 2-bit, the precision where
the paper shows the largest gap (Table 4).


In [ ]:
print("===== Ablation @ 2-bit =====")

# Step 1 only: zero fine-tuning epochs -> Gamma stays 0, S = pure Step-1 grid-search scale.
qm1 = copy.deepcopy(model).to(DEVICE)
qm1, _ = quantize_model(qm1, ref_model, calib_batches, n_bits=2, group_size=GROUP_SIZE,
                         epochs=0, lr=QUANT_LR, weight_decay=WEIGHT_DECAY, device=DEVICE,
                         compute_dtype=MODEL_DTYPE, verbose=False)
ppl1, _ = evaluate_wikitext2_ppl(qm1, test_ids, EVAL_SEQLEN, DEVICE, EVAL_MAX_CHUNKS)
print(f"Step1 only                  : perplexity={ppl1:.3f}")
del qm1
if DEVICE == "cuda":
    torch.cuda.empty_cache()

# Step 2 only: replace Algorithm-1's grid search with the naive b=1 init (s=s0),
# then fine-tune Gamma as usual -- isolates the value of the grid search itself.
_orig_init = data_agnostic_scale_init

def _naive_init(W, n_bits, group_size=128, n_candidates=200, device=None):
    return _orig_init(W, n_bits, group_size=group_size, n_candidates=1, device=device)

globals()['data_agnostic_scale_init'] = _naive_init
qm2 = copy.deepcopy(model).to(DEVICE)
qm2, _ = quantize_model(qm2, ref_model, calib_batches, n_bits=2, group_size=GROUP_SIZE,
                         epochs=EPOCHS_2BIT, lr=QUANT_LR, weight_decay=WEIGHT_DECAY,
                         device=DEVICE, compute_dtype=MODEL_DTYPE, verbose=False)
globals()['data_agnostic_scale_init'] = _orig_init
ppl2, _ = evaluate_wikitext2_ppl(qm2, test_ids, EVAL_SEQLEN, DEVICE, EVAL_MAX_CHUNKS)
print(f"Step2 only (naive b=1 init) : perplexity={ppl2:.3f}")
del qm2
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print(f"Step1 + Step2 (full method) : perplexity={results['pot_2bit']:.3f}")


## 9. Section 4: hardware-friendly bitwise dequantization (Fig. 4)

Standard dequantization needs a float multiply: `w = scale * q` (Eq. 19).
PoT dequantization instead reconstructs `w = (-1)^sign * 2^E * scale`
(Eq. 20-21). Multiplying an FP16 number by a power of two is *exactly*
the same as adding `E` to the 5-bit exponent field of its raw bit
pattern (and XOR-ing the sign bit) — this cell implements that trick
with pure integer bit ops (no float multiply) and checks it's bit-exact
against the naive formula. This part is architecture-independent (it
operates on synthetic scale/sign/exponent tensors, not the model), so
it is unchanged from the algorithmic core.


In [ ]:
import time

def dequant_naive_fp(scale, sign, exponent):
    '''Standard floating-point dequantization: w = sign * scale * 2^E.'''
    return sign * scale.to(torch.float16) * torch.pow(2.0, exponent.to(torch.float16))


def dequant_bitwise(scale, sign, exponent):
    '''PoT-PTQ style dequantization: bit manipulation + integer addition
    only (Fig. 4, Sec. 4.2), no floating-point multiply.'''
    assert scale.dtype == torch.float16
    scale = scale.contiguous()
    scale_bits = scale.view(torch.int16).to(torch.int32) & 0xFFFF

    MANTISSA_BITS = 10
    exp_shift = (exponent.to(torch.int32) & 0x1F) << MANTISSA_BITS

    orig_sign_bit = (scale_bits >> 15) & 0x1
    magnitude_bits = scale_bits & 0x7FFF
    new_magnitude = (magnitude_bits + exp_shift) & 0x7FFF   # integer addition (Sec 4.2 (b))

    pot_sign_bit = torch.where(sign < 0, torch.ones_like(orig_sign_bit), torch.zeros_like(orig_sign_bit))
    final_sign_bit = orig_sign_bit ^ pot_sign_bit            # bit manipulation (Sec 4.2 (a))

    result_bits = (new_magnitude | (final_sign_bit << 15)).to(torch.int16)
    return result_bits.view(torch.float16)


torch.manual_seed(0)
n = 500_000
scale = (torch.rand(n, device=DEVICE) * 0.05 + 1e-3).to(torch.float16)
sign = torch.where(torch.rand(n, device=DEVICE) > 0.5, 1.0, -1.0).to(torch.float16)
exponent = torch.randint(0, 4, (n,), device=DEVICE)  # 3-bit PoT -> qmax=3

w_naive = dequant_naive_fp(scale, sign, exponent)
w_bitwise = dequant_bitwise(scale, sign, exponent)
max_err = (w_naive.float() - w_bitwise.float()).abs().max().item()
print(f"Correctness: naive-vs-bitwise max abs error = {max_err:.6g} "
      f"(match={torch.allclose(w_naive.float(), w_bitwise.float())})")

def timeit(fn, iters=20):
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn(scale, sign, exponent)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters

t_naive, t_bitwise = timeit(dequant_naive_fp), timeit(dequant_bitwise)
print(f"float-multiply dequant  : {t_naive*1e3:.4f} ms")
print(f"bit-manipulation dequant: {t_bitwise*1e3:.4f} ms")
if DEVICE == "cpu":
    print("\nNote: this environment has no CUDA GPU. The paper's reported "
          "3.67x/1.63x speedups (Table 6) come from a fused CUDA kernel; "
          "on CPU integer ops don't have the same throughput edge over "
          "float multiply. The formula above is correct and GPU-ready "
          "(runs unmodified if DEVICE == 'cuda').")


## 10. (Optional) Generate text from the quantized model

Sanity check that the 3-bit/2-bit quantized ~1B model still produces
coherent text.


In [ ]:
prompt = "The history of artificial intelligence began"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)

for n_bits, qm in quantized_models.items():
    qm.eval()
    with torch.no_grad():
        out = qm.generate(input_ids, max_new_tokens=60, do_sample=True,
                           temperature=0.9, top_k=50, pad_token_id=tokenizer.pad_token_id)
    print(f"--- {n_bits}-bit POT-PTQ sample ---")
    print(tokenizer.decode(out[0], skip_special_tokens=True))
    print()
